# 01. Data And Benchmark Setup

This notebook is responsible only for the data side of the project:
- expected raw files
- loading the UD English EWT benchmark
- checking dataset statistics
- inspecting labels, samples, and OOV behavior

The goal is to keep the experimental notebook smaller and let the reviewer understand the dataset first.


## Expected Input Files

The notebook expects the following raw files in `data/raw/`:
- `en_ewt-ud-train.conllu`
- `en_ewt-ud-dev.conllu`
- `en_ewt-ud-test.conllu`

No preprocessed artifacts are required.


## Dataset Download

If the raw UD English EWT files are missing, download them first and place them into `data/raw/`.
The notebook itself does not rely on any preprocessed artifacts.


In [ ]:
# Run this cell in a terminal-enabled environment only if `data/raw/` is empty.
download_commands = """
mkdir -p data/raw
curl -L https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu -o data/raw/en_ewt-ud-train.conllu
curl -L https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu -o data/raw/en_ewt-ud-dev.conllu
curl -L https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu -o data/raw/en_ewt-ud-test.conllu
"""
print(download_commands)


In [3]:
import random
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from conllu import parse_incr


# Fixed raw filenames for the shared UD English EWT benchmark.
RAW_FILENAMES = {
    'train': 'en_ewt-ud-train.conllu',
    'dev': 'en_ewt-ud-dev.conllu',
    'test': 'en_ewt-ud-test.conllu',
}


@dataclass
class SentenceRecord:
    sentence_id: str
    tokens: list[str]
    tags: list[str]
    metadata: dict[str, Any]


def set_seed(seed: int = 13) -> None:
    # Keep randomization stable across Python, NumPy, and Torch runs.
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def normalize_neural_token(token: str) -> str:
    # Light normalization is only used by the neural model vocabulary.
    if token.isdigit():
        return '<NUM>'
    if any(char.isdigit() for char in token):
        return '<HASDIGIT>'
    return token.lower()


def load_conllu_sentences(path: str | Path, tag_field: str = 'upostag') -> list[SentenceRecord]:
    path_obj = Path(path)
    sentences: list[SentenceRecord] = []
    with path_obj.open('r', encoding='utf-8') as handle:
        for sentence_index, tokenlist in enumerate(parse_incr(handle)):
            tokens, tags = [], []
            for token in tokenlist:
                token_id = token.get('id')
                if not isinstance(token_id, int):
                    continue
                surface = token.get('form')
                tag = token.get(tag_field)
                if surface is None or tag is None:
                    continue
                tokens.append(str(surface))
                tags.append(str(tag))
            if not tokens:
                continue
            sentences.append(
                SentenceRecord(
                    sentence_id=tokenlist.metadata.get('sent_id', f'{path_obj.stem}-{sentence_index:05d}'),
                    tokens=tokens,
                    tags=tags,
                    metadata=dict(tokenlist.metadata),
                )
            )
    return sentences


def build_vocabulary(
    sentences: list[SentenceRecord],
    lowercase: bool = False,
    normalize: bool = False,
) -> Counter[str]:
    vocab: Counter[str] = Counter()
    for sentence in sentences:
        for token in sentence.tokens:
            key = normalize_neural_token(token) if normalize else token.lower() if lowercase else token
            vocab[key] += 1
    return vocab


def get_label_set(sentences: list[SentenceRecord]) -> list[str]:
    return sorted({tag for sentence in sentences for tag in sentence.tags})


def sentence_oov_rate(tokens: list[str], train_vocabulary: set[str]) -> float:
    if not tokens:
        return 0.0
    return sum(token not in train_vocabulary for token in tokens) / len(tokens)


def split_statistics(
    name: str,
    sentences: list[SentenceRecord],
    train_vocabulary: set[str] | None = None,
) -> dict[str, Any]:
    lengths = [len(sentence.tokens) for sentence in sentences]
    token_total = sum(lengths)
    tag_counter = Counter(tag for sentence in sentences for tag in sentence.tags)
    vocabulary = {token for sentence in sentences for token in sentence.tokens}
    stats: dict[str, Any] = {
        'split': name,
        'num_sentences': len(sentences),
        'num_tokens': token_total,
        'avg_sentence_length': round(float(np.mean(lengths)), 3),
        'median_sentence_length': round(float(np.median(lengths)), 3),
        'max_sentence_length': max(lengths),
        'vocabulary_size': len(vocabulary),
        'num_tags': len(tag_counter),
    }
    if train_vocabulary is not None:
        oov_rates = [sentence_oov_rate(sentence.tokens, train_vocabulary) for sentence in sentences]
        flat_oov = sum(token not in train_vocabulary for sentence in sentences for token in sentence.tokens)
        stats['token_oov_rate'] = round(flat_oov / token_total, 5)
        stats['avg_sentence_oov_rate'] = round(float(np.mean(oov_rates)), 5)
    return stats


def load_benchmark(raw_dir: str | Path = 'data/raw', tag_field: str = 'upostag') -> dict[str, list[SentenceRecord]]:
    raw_dir_path = Path(raw_dir)
    return {
        split: load_conllu_sentences(raw_dir_path / filename, tag_field=tag_field)
        for split, filename in RAW_FILENAMES.items()
    }


## Load The Benchmark

This step reads the raw `.conllu` files, constructs the train/dev/test split in memory, and computes split-level statistics.


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TAG_FIELD = 'upostag'
SEED = 13

set_seed(SEED)
dataset = load_benchmark(raw_dir=RAW_DIR, tag_field=TAG_FIELD)
train_vocabulary = set(build_vocabulary(dataset['train']).keys())
labels = get_label_set(dataset['train'])
stats_df = pd.DataFrame([
    split_statistics(split, rows, train_vocabulary if split != 'train' else None)
    for split, rows in dataset.items()
])

stats_df


,split,num_sentences,num_tokens,avg_sentence_length,median_sentence_length,max_sentence_length,vocabulary_size,num_tags,token_oov_rate,avg_sentence_oov_rate
0,train,12544,204577,16.309,14.0,159,19674,17,NaN,NaN
1,dev,2001,25147,12.567,10.0,75,5494,17,0.08303,0.12843
2,test,2077,25094,12.082,9.0,81,5629,17,0.09134,0.13837


In [5]:
print('Tag field:', TAG_FIELD)
print('Number of labels:', len(labels))
print('Labels:', labels)


Tag field: upostag
Number of labels: 17
Labels: ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


## Benchmark Takeaway

The benchmark is suitable for comparative error analysis because it is large enough to be realistic, but still compact enough to inspect in detail. The `dev` and `test` splits contain non-trivial OOV pressure, which makes the evaluation more informative than a purely in-vocabulary comparison.


## Inspect Sample Sentences

A quick sample helps verify that tokenization and POS labels were loaded correctly before any model training starts.


In [6]:
for split_name in ['train', 'dev', 'test']:
    sample = dataset[split_name][0]
    print(f'\nSplit: {split_name}')
    print('Sentence ID:', sample.sentence_id)
    print('Tokens:', sample.tokens[:20])
    print('Tags:', sample.tags[:20])



Split: train
Sentence ID: weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001
Tokens: ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the']
Tags: ['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET']

Split: dev
Sentence ID: weblog-blogspot.com_nominations_20041117172713_ENG_20041117_172713-0001
Tokens: ['From', 'the', 'AP', 'comes', 'this', 'story', ':']
Tags: ['ADP', 'DET', 'PROPN', 'VERB', 'DET', 'NOUN', 'PUNCT']

Split: test
Sentence ID: weblog-blogspot.com_zentelligence_20040423000200_ENG_20040423_000200-0001
Tokens: ['What', 'if', 'Google', 'Morphed', 'Into', 'GoogleOS', '?']
Tags: ['PRON', 'SCONJ', 'PROPN', 'VERB', 'ADP', 'PROPN', 'PUNCT']


## Inspect OOV Pressure

This gives a quick sense of how different the dev/test vocabulary is from the train vocabulary.


In [7]:
for split_name in ['dev', 'test']:
    rates = [sentence_oov_rate(sentence.tokens, train_vocabulary) for sentence in dataset[split_name]]
    print(split_name, 'mean sentence OOV rate =', round(float(np.mean(rates)), 4))
    print(split_name, 'max sentence OOV rate  =', round(float(np.max(rates)), 4))


dev mean sentence OOV rate = 0.1284
dev max sentence OOV rate  = 1.0
test mean sentence OOV rate = 0.1384
test max sentence OOV rate  = 1.0


## Data-Level Interpretation

Two properties of this benchmark are especially important for the later analysis:

- `test` has slightly higher OOV pressure than `dev`, so generalization to unseen words matters.
- The combination of web text, named entities, and lexical variation makes noun/proper-noun ambiguity a natural difficulty source even before model training begins.


# Available Model Notebooks

The benchmark notebook is followed by four model-specific notebooks:
- `02_averaged_perceptron.ipynb`
- `03_hmm_viterbi.ipynb`
- `04_crf.ipynb`
- `05_compact_bilstm.ipynb`
